If running ONLINE make sure to upload the predictions accordingly. The ones we used can be downloaded and have final in their name. Note that one prediction for model is required.

In [ ]:
ONLINE = False
use_final = False # ensemble on the provided predictions
predictions_path = '.' if ONLINE else 'predictions'

In [ ]:
import numpy as np
import pandas as pd
import gdown
import os
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

In [ ]:
# Download training and submission data
if ONLINE:
    from google.colab import files
    url = 'https://drive.google.com/file/d/1lDEtOq644qlaw_vh0i5c3ChVfrHJiuMc/view?usp=sharing'
    output = 'data_train.csv'
    gdown.download(url, output, fuzzy=True)
    url = 'https://drive.google.com/file/d/1KMiakfzTWYr4QJU9S5DC9e5KIo_8bb24/view?usp=sharing'
    output = 'sampleSubmission.csv'
    gdown.download(url, output, fuzzy=True)

    uploaded = files.upload() # upload the predictions


In [ ]:
models = ['ae', 'als', 'jncf', 'gcn', 'svd', 'bmf', 'bmf_reg', 'bmf_ord']

predictions_filenames = []

for model in models:
    if use_final:
        predictions_filenames.append(os.path.join(predictions_path, f"final_{model}_predictions.npz"))
    else:
        predictions_filenames.append(os.path.join(predictions_path, f"{model}_predictions.npz"))

In [ ]:
def load_predictions(filenames):
    test_predictions, challenge_predictions = [], []
    for file in filenames:
        loaded_predictions = np.load(file)
        test_predictions.append(loaded_predictions['test'].reshape(-1, 1))
        challenge_predictions.append(loaded_predictions['challenge'].reshape(-1, 1))
    return test_predictions, challenge_predictions

def extract_predictions(data_pd):
    return data_pd.Prediction.values

In [ ]:
# @title Load predictions and targets
data_pd = pd.read_csv('data_train.csv')
submission_pd = pd.read_csv('sampleSubmission.csv')

# Split the dataset into train and test
train_size = 0.9
_, test_pd = train_test_split(data_pd, train_size=train_size, random_state=42)

test_predictions = extract_predictions(test_pd)

models_test_predictions, models_challenge_predictions = load_predictions(predictions_filenames)

meta_test_predictions = np.hstack(models_test_predictions)
meta_challenge_predictions = np.hstack(models_challenge_predictions)

In [ ]:
# @title Training
blender = LinearRegression().fit(meta_test_predictions, test_predictions)

In [ ]:
# @title Generate submission
final_predictions = blender.predict(meta_challenge_predictions)

# Predict the ratings using the trained model
submission_pd['Prediction'] = final_predictions.clip(1, 5)

# Save the predictions to a new CSV file
submission_pd[['Id', 'Prediction']].to_csv('submission.csv', index=False)